In [1]:
import json
import pandas as pd
from src import data_cleaner
from src.data_validator import generate_quality_report
from src.data_validator import load_rules
from src.data_loader import load_csv
from src.data_validator import check_value_range


In [2]:
path = r"C:\Users\Meta Pc\PycharmProjects\DataSentry\data\dirty_sample.csv"
df = load_csv(path)

In [3]:
df

,product,units_sold,unit_price,region
0,Laptop,5,1200.0,Berlin
1,Mouse,20,25.0,Paris
2,Keyboard,12,75.0,Berlin
3,Monitor,8,300.0,London
4,Laptop,6,1200.0,Paris
5,Mouse,18,NaN,London
6,Keyboard,15,75.0,Berlin
7,Monitor,10,300.0,Paris
8,Laptop,unknown,1200.0,London
9,Mouse,25,25.0,Berlin


In [61]:
with open("../config/data_rules.json", "r") as f:
    j = json.load(f)
j


{'allowed_categories': {'region': ['Berlin', 'Paris', 'London'],
  'product': ['Laptop', 'Mouse', 'Keyboard', 'Monitor']},
 'value_ranges': {'unit_price': {'min': 0, 'max': 1000},
  'units_sold': {'min': 0, 'max': 1000}},
 'required_columns': ['product', 'units_sold', 'unit_price', 'region'],
 'required_columns_type': {'units_sold': 'numeric',
  'unit_price': 'numeric',
  'product': 'string'}}

In [3]:
rules = load_rules("../config/data_rules.json")
def run_quality_pipeline(path, columns:list, rules):
    df = load_csv(path)
    initial_report = generate_quality_report(df, columns, rules)
    df_copy = df.copy()
    df_copy = data_cleaner.remove_duplicates(df_copy)
    df_copy = data_cleaner.handle_missing_values(df_copy)
    df_copy = data_cleaner.handle_invalid_numeric(df_copy)
    final_report = generate_quality_report(df_copy, columns, rules)
    return (df, initial_report, df_copy, final_report )


In [5]:
df, report, clean, final_report  = run_quality_pipeline(path, ["units_sold","unit_price"], rules)

In [6]:
df

,product,units_sold,unit_price,region
0,Laptop,5,1200.0,Berlin
1,Mouse,20,25.0,Paris
2,Keyboard,12,75.0,Berlin
3,Monitor,8,300.0,London
4,Laptop,6,1200.0,Paris
5,Mouse,18,NaN,London
6,Keyboard,15,75.0,Berlin
7,Monitor,10,300.0,Paris
8,Laptop,unknown,1200.0,London
9,Mouse,25,25.0,Berlin


In [7]:
clean

,product,units_sold,unit_price,region
0,Laptop,5.0,1200.0,Berlin
1,Mouse,20.0,25.0,Paris
2,Keyboard,12.0,75.0,Berlin
3,Monitor,8.0,300.0,London
4,Laptop,6.0,1200.0,Paris
5,Mouse,18.0,25.0,London
6,Keyboard,15.0,75.0,Berlin
7,Monitor,10.0,300.0,Paris
9,Mouse,25.0,25.0,Berlin
10,Keyboard,11.0,75.0,Atlantis


In [8]:
report

{'missing_columns': [],
 'incorrect_types': [['units_sold', 'numeric']],
 'missing_values_percentage': np.float64(1.9230769230769231),
 'duplicate_percentage': np.float64(7.6923076923076925),
 'missing_values_count': np.int64(1),
 'duplicate_count': np.int64(1),
 'allowed_categories': {},
 'wrong_categories': {},
 'invalid_numeric_count': {'units_sold': 1, 'unit_price': 0},
 'invalid_numeric_percentage': {'units_sold': 7.6923076923076925,
  'unit_price': 0.0},
 'quality_score': np.float64(95.51282051282051)}

In [9]:
final_report

{'missing_columns': [],
 'incorrect_types': [],
 'missing_values_percentage': np.float64(0.0),
 'duplicate_percentage': np.float64(0.0),
 'missing_values_count': np.int64(0),
 'duplicate_count': np.int64(0),
 'allowed_categories': {},
 'wrong_categories': {},
 'invalid_numeric_count': {'units_sold': 0, 'unit_price': 0},
 'invalid_numeric_percentage': {'units_sold': 0.0, 'unit_price': 0.0},
 'quality_score': np.float64(100.0)}

In [65]:
j["allowed_categories"]["product"]

['Laptop', 'Mouse', 'Keyboard', 'Monitor']

In [66]:
j["value_ranges"]["units_sold"]["min"]

0

In [67]:
columns = ["region", "product"]
rules = load_rules("../config/data_rules.json")
print(generate_quality_report(df, columns, rules))

{'missing_columns': [], 'incorrect_types': [['units_sold', 'numeric']], 'invalid_unit_price': 4, 'invalid_units_sold': 0, 'unit_price_error_percentage': 30.76923076923077, 'units_sold_error_percentage': 0.0, 'missing_values_percentage': np.float64(1.9230769230769231), 'duplicate_percentage': np.float64(7.6923076923076925), 'missing_values_count': np.int64(1), 'duplicate_count': np.int64(1), 'allowed_categories': {'region': ['Berlin', 'Paris', 'London'], 'product': ['Laptop', 'Mouse', 'Keyboard', 'Monitor']}, 'wrong_categories': {'region': region
Atlantis    1
Name: count, dtype: int64, 'product': Series([], Name: count, dtype: int64)}}


In [68]:
print(generate_quality_report(df, ["region", "product"], rules))

{'missing_columns': [], 'incorrect_types': [['units_sold', 'numeric']], 'invalid_unit_price': 4, 'invalid_units_sold': 0, 'unit_price_error_percentage': 30.76923076923077, 'units_sold_error_percentage': 0.0, 'missing_values_percentage': np.float64(1.9230769230769231), 'duplicate_percentage': np.float64(7.6923076923076925), 'missing_values_count': np.int64(1), 'duplicate_count': np.int64(1), 'allowed_categories': {'region': ['Berlin', 'Paris', 'London'], 'product': ['Laptop', 'Mouse', 'Keyboard', 'Monitor']}, 'wrong_categories': {'region': region
Atlantis    1
Name: count, dtype: int64, 'product': Series([], Name: count, dtype: int64)}}


In [69]:
print(check_value_range(df, "unit_price", rules["value_ranges"]["unit_price"]))



4


In [70]:

print(check_value_range(df, "units_sold", rules["value_ranges"]["units_sold"]))

0


In [1]:
from src.quality_pipeline import run_quality_pipeline, generate_cleaning_summary, generate_quality_summary,  generate_final_report
from src.data_validator import load_rules

rules = load_rules("../config/data_rules.json")

df, initial_report, cleaned_df, final_report, cleaning_summary = run_quality_pipeline(
    "../data/dirty_sample.csv",
    ["region", "product"],
    rules
)

report = generate_cleaning_summary(cleaning_summary)

print(report)

{'missing_columns': [], 'incorrect_types': [['units_sold', 'numeric']], 'missing_values_percentage': np.float64(1.9230769230769231), 'duplicate_percentage': np.float64(7.6923076923076925), 'missing_values_count': np.int64(1), 'duplicate_count': np.int64(1), 'allowed_categories': {'region': ['Berlin', 'Paris', 'London'], 'product': ['Laptop', 'Mouse', 'Keyboard', 'Monitor']}, 'wrong_categories': {'region': region
Atlantis    1
Name: count, dtype: int64, 'product': Series([], Name: count, dtype: int64)}, 'invalid_numeric_count': {'units_sold': 1, 'unit_price': 0}, 'invalid_numeric_percentage': {'units_sold': 7.6923076923076925, 'unit_price': 0.0}, 'quality_score': np.float64(95.67307692307693)}
{'missing_columns': [], 'incorrect_types': [], 'missing_values_percentage': np.float64(0.0), 'duplicate_percentage': np.float64(0.0), 'missing_values_count': np.int64(0), 'duplicate_count': np.int64(0), 'allowed_categories': {'region': ['Berlin', 'Paris', 'London'], 'product': ['Laptop', 'Mouse', 

In [2]:
quality_report = generate_quality_summary(initial_report)
print(quality_report)

DataSentry Quality Report
-------------------------
Quality Score: 95.67
Missing Values: 1
Duplicate Rows: 1
Invalid Categories: 1
Invalid Numeric Values: 1
Missing Columns: 0
Incorrect Column Types: 1



In [3]:
final_report_text = generate_final_report(
    initial_report,
    final_report,
    cleaning_summary
)

print(final_report_text)

DATASENTRY REPORT
[INITIAL DATA QUALITY]
DataSentry Quality Report
-------------------------
Quality Score: 95.67
Missing Values: 1
Duplicate Rows: 1
Invalid Categories: 1
Invalid Numeric Values: 1
Missing Columns: 0
Incorrect Column Types: 1


[CLEANING SUMMARY]
DataSentry Cleaning Report
--------------------------
Original Rows: 13
Duplicates Removed: 1
Rows After Duplicate Removal: 12
Rows Removed During Missing Value Handling: 0
Rows After Missing Value Handling: 12
Invalid Numeric Rows Removed: 1
Final Rows: 11
Total Rows Removed: 2

[FINAL DATA QUALITY]
DataSentry Quality Report
-------------------------
Quality Score: 98.86
Missing Values: 0
Duplicate Rows: 0
Invalid Categories: 1
Invalid Numeric Values: 0
Missing Columns: 0
Incorrect Column Types: 0




In [4]:
from src import data_cleaner
from src.data_validator import load_rules
rules  = load_rules("../config/data_rules.json")

normalized_df = data_cleaner.normalize_categories(df, rules)

print("Original:", df["product"].unique())
print("Normalized:", normalized_df["product"].unique())

print("Regions:", normalized_df["region"].unique())

Original: <ArrowStringArray>
['Laptop', 'Mouse', 'Keyboard', 'Monitor']
Length: 4, dtype: str
Normalized: <ArrowStringArray>
['Laptop', 'Mouse', 'Keyboard', 'Monitor']
Length: 4, dtype: str
Regions: <ArrowStringArray>
['Berlin', 'Paris', 'London', 'Atlantis']
Length: 4, dtype: str


In [5]:
test_df = df.copy()
test_df.loc[test_df.index[0], "product"] = "keyboard"

normalized_test = data_cleaner.normalize_categories(test_df, rules)

print("Before:", test_df["product"].unique())
print("After:", normalized_test["product"].unique())

Before: <ArrowStringArray>
['keyboard', 'Mouse', 'Keyboard', 'Monitor', 'Laptop']
Length: 5, dtype: str
After: <ArrowStringArray>
['Keyboard', 'Mouse', 'Monitor', 'Laptop']
Length: 4, dtype: str


In [8]:
from src.quality_pipeline import run_quality_pipeline
from src.quality_pipeline import generate_final_report

rules = load_rules("../config/data_rules.json")

df, initial_report, cleaned_df, final_report, cleaning_summary = run_quality_pipeline(
    "../data/dirty_sample.csv",
    ["product", "region"],
    rules
)

print(generate_final_report(initial_report, final_report, cleaning_summary))

df, initial_report, cleaned_df, final_report, cleaning_summary = run_quality_pipeline(
    "../data/dirty_sample.csv",
    ["product", "region"],
    rules
)

print("Initial:", initial_report["out_of_range_count"])
print("Final:", final_report["out_of_range_count"])

DATASENTRY REPORT
[INITIAL DATA QUALITY]
DataSentry Quality Report
-------------------------
Quality Score: 93.46
Missing Values: 1
Duplicate Rows: 1
Invalid Categories: 1
Invalid Numeric Values: 1
Missing Columns: 0
Incorrect Column Types: 1


[CLEANING SUMMARY]
DataSentry Cleaning Report
--------------------------
Original Rows: 13
Duplicates Removed: 1
Rows After Duplicate Removal: 12
Rows Removed During Missing Value Handling: 0
Rows After Missing Value Handling: 12
Invalid Numeric Rows Removed: 1
Categories Normalized: 0
Final Rows: 11
Total Rows Removed: 2

[FINAL DATA QUALITY]
DataSentry Quality Report
-------------------------
Quality Score: 97.27
Missing Values: 0
Duplicate Rows: 0
Invalid Categories: 1
Invalid Numeric Values: 0
Missing Columns: 0
Incorrect Column Types: 0


Initial: {'unit_price': 4, 'units_sold': 0}
Final: {'unit_price': 2, 'units_sold': 0}


In [4]:
from src.data_validator import generate_quality_report
from src.quality_pipeline import load_rules

# Load rules
rules = load_rules("../config/data_rules.json")

# Load dirty dataset
df = pd.read_csv("../data/dirty_sample.csv")

# Generate quality report
report = generate_quality_report(
    df,
    ["product", "region"],
    rules
)

# Check Out of Range Validation
print("Out of Range Count:")
print(report["out_of_range_count"])

print("\nOut of Range Percentage:")
print(report["out_of_range_percentage"])

# Check Quality Score
print("\nQuality Score:")
print(report["quality_score"])

Out of Range Count:
{'unit_price': 4, 'units_sold': 0}

Out of Range Percentage:
{'unit_price': 30.76923076923077, 'units_sold': 0.0}

Quality Score:
93.46153846153848
